# Gemini Exemplar Dataset Audit

This notebook audits the **training split only** to identify candidate whole reflections and skill-band/micro-label segments for expert review as possible Gemini prompt examples. It checks row, segment, and document consistency; it does not decide semantic annotation correctness or generate explanations. No Gemini or other provider calls are made.

The source contains controlled reflection text and stable identifiers. Reflection text is not displayed by default. Audit previews and reports omit raw identifiers. Full-text review is opt-in and should be performed locally; clear outputs before saving or sharing this notebook. Exports are disabled by default and, when enabled, contain aggregate findings only in approved protected storage.

Training labels are candidate-grain. Multiple candidate bundles for one segment are retained, not averaged or arbitrarily collapsed. Documents with label disagreement are excluded from the whole-reflection shortlist; segment-level micro-label sampling additionally requires a single, non-review micro-label candidate whose text and broad band match the audited training segment.

## 1. Load Configuration and Libraries

Set the review and export switches deliberately. `REVIEW_TEXT` remains false for routine audits; use session-local shortlist references in `REVIEW_REFS` only after the aggregate checks pass. This notebook reads only the canonical training modeling view.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
assert (PROJECT_ROOT / "config" / "experiment.toml").is_file(), "Run from the prebi_v1 project root."
load_dotenv(PROJECT_ROOT / ".env", override=False)
SPLIT_DIR = PROJECT_ROOT / "data" / "provisional_segment_splits"
TRAIN_PATH = SPLIT_DIR / "provisional_train_modeling_wide.csv"

MISSINGNESS_THRESHOLD = 0.05
MAX_SHORTLIST_DOCUMENTS = 12
REVIEW_TEXT = False
REVIEW_REFS = []
EXPORT_AGGREGATES = False
PROTECTED_ROOT_VALUE = os.getenv("PREBI_DATA_ROOT")

DIMENSIONS = {
    "SW": "target_situationserfassung",
    "UA": "target_analyse",
    "HA": "target_konsequenzen",
}
LABEL_COLUMNS = ["scope_target", *DIMENSIONS.values()]
TEXT_QUALITY_FLAGS = [
    "requires_review", "text_missing_or_blank", "text_punctuation_only",
    "text_too_short", "text_quality_excluded",
]
EXPECTED_TYPES = {
    "document_id": "string", "segment_id": "string", "candidate_id": "string",
    "segment_order": "integer", "scope_target": "integer-like",
}

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (9, 4)
print("Configuration loaded. Provider calls are not used by this notebook.")

Configuration loaded. Provider calls are not used by this notebook.


## 2. Ingest Dataset and Create Working Copy

Load the approved training export once, preserve an unchanged in-memory raw snapshot, and audit a separate working copy. Do not write a raw-data copy into the project tree.

In [2]:
required_columns = {
    "candidate_id", "document_id", "segment_id", "segment_order", "text",
    "split", "scope_target", *DIMENSIONS.values(), *TEXT_QUALITY_FLAGS,
}
if not TRAIN_PATH.is_file():
    raise FileNotFoundError("Canonical training modeling view was not found.")
raw_snapshot = pd.read_csv(
    TRAIN_PATH,
    dtype={"candidate_id": "string", "document_id": "string", "segment_id": "string"},
)
missing_columns = required_columns - set(raw_snapshot.columns)
if missing_columns:
    raise ValueError(f"Training view is missing required audit columns: {sorted(missing_columns)}")
working = raw_snapshot.copy(deep=True)
if not working["split"].eq("train").all():
    raise ValueError("The selected source contains rows outside the training split.")
if working.empty:
    raise ValueError("Training view is empty.")

print(f"Loaded {len(working):,} candidate rows across {working['segment_id'].nunique():,} segments and {working['document_id'].nunique():,} documents.")
print(f"Column count: {working.shape[1]}. Raw snapshot retained in memory; transformations use a separate working copy.")

Loaded 7,107 candidate rows across 7,065 segments and 181 documents.
Column count: 34. Raw snapshot retained in memory; transformations use a separate working copy.


## 3. Preview Rows and Validate Column Types

The preview excludes reflection text and identifiers. Candidate rows remain distinct; repeated segments are not automatically duplicates. Inspect the expected-type checks before interpreting downstream profiles.

In [3]:
def expected_type_status(frame: pd.DataFrame) -> pd.DataFrame:
    checks = []
    for column, expected in EXPECTED_TYPES.items():
        series = frame[column]
        if expected == "string":
            matches = pd.api.types.is_string_dtype(series.dtype)
        elif expected == "integer":
            matches = pd.api.types.is_integer_dtype(series.dtype)
        else:
            values = pd.to_numeric(series, errors="coerce")
            matches = values.notna().all() and np.equal(values, np.floor(values)).all()
        checks.append({"column": column, "expected": expected, "observed_dtype": str(series.dtype), "match": bool(matches)})
    return pd.DataFrame(checks)

schema_report = expected_type_status(working)
display(schema_report)
preview_columns = [
    "component", "scope_target", "target_situationserfassung",
    "target_analyse", "target_konsequenzen", "requires_review",
]
display(working[preview_columns].head(8))
print("Preview is label-only; no text or row identifiers are displayed.")

,column,expected,observed_dtype,match
0,document_id,string,string,True
1,segment_id,string,string,True
2,candidate_id,string,string,True
3,segment_order,integer,int64,True
4,scope_target,integer-like,int64,True


,component,scope_target,target_situationserfassung,target_analyse,target_konsequenzen,requires_review
0,in_scope,1,0.0,1.0,0.0,False
1,in_scope,1,0.0,1.0,0.0,False
2,in_scope,1,0.0,1.0,0.0,False
3,in_scope,1,0.0,0.0,2.0,False
4,in_scope,1,0.0,1.0,0.0,False
5,in_scope,1,0.0,2.0,0.0,False
6,in_scope,1,1.0,0.0,0.0,False
7,in_scope,1,0.0,2.0,0.0,False


Preview is label-only; no text or row identifiers are displayed.


## 4. Profile Missing Values and Null Patterns

Report missingness by field and plot only column-level percentages. Missing skill labels are expected for out-of-scope rows; the rule checks account for that rather than treating them as missing annotation failures.

In [5]:
missingness = pd.DataFrame({
    "missing_count": working.isna().sum(),
    "missing_percent": working.isna().mean().mul(100),
}).sort_values("missing_percent", ascending=False)
conditionally_nullable = {
    "text_quality_exclusion_reason", "category", "level", "band", "band_code",
    *DIMENSIONS.values(),
}
missingness["missingness_type"] = [
    "conditionally nullable" if column in conditionally_nullable else "expected required"
    for column in missingness.index
]
missingness["over_threshold"] = (
    missingness["missing_percent"].ge(MISSINGNESS_THRESHOLD * 100)
    & missingness["missingness_type"].eq("expected required")
)
display(missingness.round(2))
unexpected_missingness = missingness.loc[
    missingness["missingness_type"].eq("expected required") & missingness["missing_count"].gt(0),
    "missing_percent",
].sort_values()
if unexpected_missingness.empty:
    print("No unexpected missing values to plot.")
else:
    ax = unexpected_missingness.plot.barh(color="#39796B")
    ax.axvline(MISSINGNESS_THRESHOLD * 100, color="#b34d4d", linestyle="--", label="configured threshold")
    ax.set(xlabel="Missing values (%)", ylabel="", title="Unexpected column-level missingness")
    ax.legend(frameon=False)
    plt.tight_layout()
    plt.show()
print("Skill-label nulls for out-of-scope rows and unused rationale fields are expected by the dataset contract.")

,missing_count,missing_percent,missingness_type,over_threshold
text_quality_exclusion_reason,7107,100.00,conditionally nullable,False
target_analyse,342,4.81,conditionally nullable,False
category,342,4.81,conditionally nullable,False
target_situationserfassung,342,4.81,conditionally nullable,False
band_code,342,4.81,conditionally nullable,False
band,342,4.81,conditionally nullable,False
level,342,4.81,conditionally nullable,False
target_konsequenzen,342,4.81,conditionally nullable,False
known_situationserfassung,0,0.00,expected required,False
known_analyse,0,0.00,expected required,False


No unexpected missing values to plot.
Skill-label nulls for out-of-scope rows and unused rationale fields are expected by the dataset contract.


## 5. Detect Duplicates and Key Uniqueness Issues

Candidate IDs should be unique. Repeated segment IDs are expected when alternative candidate bundles exist; the audit distinguishes those from exact row duplication and validates that candidate copies agree on the segment text and source order.

In [6]:
candidate_id_duplicates = int(working["candidate_id"].duplicated(keep=False).sum())
full_row_duplicate_count = int(working.duplicated().sum())
segment_counts = working.groupby(["document_id", "segment_id"], dropna=False).size()
text_variants = working.groupby(["document_id", "segment_id"], dropna=False)["text"].nunique(dropna=False)
order_variants = working.groupby(["document_id", "segment_id"], dropna=False)["segment_order"].nunique(dropna=False)
duplicate_summary = pd.DataFrame([
    {"check": "duplicate candidate_id rows", "affected_rows": candidate_id_duplicates},
    {"check": "exact duplicate rows", "affected_rows": full_row_duplicate_count},
    {"check": "segments with multiple candidate rows", "affected_rows": int(segment_counts.gt(1).sum())},
    {"check": "segments with conflicting text copies", "affected_rows": int(text_variants.gt(1).sum())},
    {"check": "segments with conflicting order copies", "affected_rows": int(order_variants.gt(1).sum())},
])
display(duplicate_summary)
if candidate_id_duplicates:
    print("Candidate key violation detected; duplicated identifiers are withheld from display.")

,check,affected_rows
0,duplicate candidate_id rows,0
1,exact duplicate rows,0
2,segments with multiple candidate rows,42
3,segments with conflicting text copies,0
4,segments with conflicting order copies,0


## 6. Audit Numeric Columns (Ranges, Outliers, Invalid Values)

Label values have explicit domains: scope is 0/1 and in-scope skill bands are 0-3. Text length and segment order are numeric metadata; IQR flags are prompts for review, not automatic grounds to discard a valid reflection.

In [13]:
numeric_columns = [
    column for column in ("segment_order", "text_length", "minimum_text_characters")
    if column in working.columns and working[column].notna().any()
]
numeric_profile = working[numeric_columns].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T if numeric_columns else pd.DataFrame()
display(numeric_profile.round(2))

scope_values = pd.to_numeric(working["scope_target"], errors="coerce")
invalid_scope = scope_values.isna() | ~scope_values.isin([0, 1])
in_scope = scope_values.eq(1)
invalid_bands = pd.Series(False, index=working.index)
for column in DIMENSIONS.values():
    bands = pd.to_numeric(working[column], errors="coerce")
    invalid_bands |= in_scope & (bands.isna() | ~bands.isin([0, 1, 2, 3]))
    invalid_bands |= scope_values.eq(0) & bands.notna()

range_findings = pd.DataFrame([
    {"rule": "scope_target in {0, 1}", "affected_rows": int(invalid_scope.sum())},
    {"rule": "in-scope skill bands in {0, 1, 2, 3}; out-of-scope bands null", "affected_rows": int(invalid_bands.sum())},
    {"rule": "segment_order is a nonnegative integer", "affected_rows": int((pd.to_numeric(working["segment_order"], errors="coerce").isna() | (pd.to_numeric(working["segment_order"], errors="coerce") < 0) | (pd.to_numeric(working["segment_order"], errors="coerce") % 1 != 0)).sum())},
])
display(range_findings)
if "text_length" in working:
    values = pd.to_numeric(working["text_length"], errors="coerce")
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    text_length_outlier_count = int(((values < q1 - 1.5 * iqr) | (values > q3 + 1.5 * iqr)).sum())
    print(f"IQR-flagged text lengths: {text_length_outlier_count:,} rows; flags are review prompts only.")
print("Ordinal scope/band labels are validated categorically, not summarized as numeric averages.")

,count,mean,std,min,1%,25%,50%,75%,99%,max
segment_order,7107.0,29.72,22.25,1.0,1.0,13.0,25.0,42.0,102.00,156.0
text_length,7107.0,149.11,155.15,10.0,38.0,96.0,133.0,180.0,373.94,7581.0
minimum_text_characters,7107.0,10.00,0.00,10.0,10.0,10.0,10.0,10.0,10.00,10.0


,rule,affected_rows
0,"scope_target in {0, 1}",0
1,"in-scope skill bands in {0, 1, 2, 3}; out-of-s...",0
2,segment_order is a nonnegative integer,0


IQR-flagged text lengths: 183 rows; flags are review prompts only.
Ordinal scope/band labels are validated categorically, not summarized as numeric averages.


## 7. Audit Categorical Columns (Cardinality, Rare/Unexpected Labels)

Show allowed-value checks and candidate-grain frequencies. Label frequency is also summarized later at unique-segment grain; do not confuse the two when segments have alternative candidate annotations.

In [20]:
categorical_columns = [
    column for column in ["component", "category", "level", "band_code", "split"]
    if column in working.columns
]
cardinality = pd.DataFrame([
    {"column": column, "distinct_values": int(working[column].nunique(dropna=False)), "high_cardinality": bool(working[column].nunique(dropna=False) > 50)}
    for column in categorical_columns
])
display(cardinality)
RARE_CATEGORY_MIN_COUNT = 10
rare_categories = []
for column in categorical_columns:
    frequencies = working[column].value_counts(dropna=False)
    for value, count in frequencies.items():
        if count < RARE_CATEGORY_MIN_COUNT:
            rare_categories.append({"column": column, "value": "<NULL>" if pd.isna(value) else str(value), "candidate_rows": int(count)})
    print(f"{column} frequencies")
    display(frequencies.rename_axis("value").to_frame("candidate_rows"))
display(pd.DataFrame(rare_categories, columns=["column", "value", "candidate_rows"]))

allowed_label_checks = {
    "scope_target": {0, 1},
    **{column: {0, 1, 2, 3} for column in DIMENSIONS.values()},
}
category_findings = []
for column, allowed in allowed_label_checks.items():
    observed = set(pd.to_numeric(working[column], errors="coerce").dropna().unique())
    unexpected = observed - allowed
    category_findings.append({"column": column, "unexpected_distinct_values": len(unexpected), "unexpected_values": sorted(unexpected)})
display(pd.DataFrame(category_findings))
print(f"Rare category threshold: fewer than {RARE_CATEGORY_MIN_COUNT} candidate rows.")

,column,distinct_values,high_cardinality
0,component,2,False
1,category,4,False
2,level,6,False
3,band_code,4,False
4,split,1,False


component frequencies


,candidate_rows
value,
in_scope,6765
out_of_scope,342


category frequencies


,candidate_rows
value,
analyse,2928
situationserfassung,2785
konsequenzen,1052
NaN,342


...

split frequencies


,candidate_rows
value,
train,7107


,column,value,candidate_rows


,column,unexpected_distinct_values,unexpected_values
0,scope_target,0,[]
1,target_situationserfassung,0,[]
2,target_analyse,0,[]
3,target_konsequenzen,0,[]


Rare category threshold: fewer than 10 candidate rows.


## 8. Check Date/Time Fields (Parsing, Gaps, Ordering)

Inspect date-like columns if the export has any. The current modeling schema is expected to contain no event timestamps, so temporal ordering is not inferred from row order; segment source order is checked explicitly in the rule-based audit.

In [11]:
import re

date_pattern = re.compile(r"(^|_)(date|datetime|timestamp|time)(_|$)")
date_columns = [column for column in working.columns if date_pattern.search(column.lower())]
if not date_columns:
    print("No date/time columns found in the canonical modeling view.")
else:
    date_audit = []
    for column in date_columns:
        parsed = pd.to_datetime(working[column], errors="coerce", utc=True)
        date_audit.append({
            "column": column,
            "invalid_non_null_values": int((working[column].notna() & parsed.isna()).sum()),
            "minimum_utc": parsed.min(),
            "maximum_utc": parsed.max(),
            "duplicate_timestamps": int(parsed.duplicated(keep=False).sum()),
        })
    display(pd.DataFrame(date_audit))

orders_per_document = working.groupby("document_id")["segment_order"].apply(
    lambda values: pd.to_numeric(values, errors="coerce").drop_duplicates().sort_values().tolist()
)
print(f"Source-order sequences inspected for {len(orders_per_document):,} training documents.")

No date/time columns found in the canonical modeling view.
Source-order sequences inspected for 181 training documents.


## 9. Run Rule-Based Data Quality Tests

Executable checks enforce train-only membership, candidate-key uniqueness, usable text, stable text/order across candidate copies, label-domain rules, and one complete label bundle per segment. A document is automatically eligible only if every segment passes. This is a consistency screen, not semantic adjudication.

In [14]:
def as_bool(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series.dtype):
        return series.fillna(False)
    return series.astype("string").str.strip().str.lower().isin({"1", "true", "yes", "y"})

for column in TEXT_QUALITY_FLAGS:
    working[f"_flag_{column}"] = as_bool(working[column])

scope_numeric = pd.to_numeric(working["scope_target"], errors="coerce")
valid_scope = scope_numeric.isin([0, 1])
structural_valid = valid_scope & working["text"].notna() & working["text"].astype("string").str.strip().ne("")
for column in TEXT_QUALITY_FLAGS:
    if column != "requires_review":
        structural_valid &= ~working[f"_flag_{column}"]
component_matches_scope = working["component"].astype("string").eq(
    scope_numeric.map({0: "out_of_scope", 1: "in_scope"}).astype("string")
)
structural_valid &= component_matches_scope
for target_column in DIMENSIONS.values():
    numeric_target = pd.to_numeric(working[target_column], errors="coerce")
    valid_target = (
        (scope_numeric.eq(1) & numeric_target.isin([0, 1, 2, 3]))
        | (scope_numeric.eq(0) & numeric_target.isna())
    )
    structural_valid &= valid_target
working["_structural_valid"] = structural_valid
working["_row_valid"] = structural_valid & ~working["_flag_requires_review"]

def label_token(value: object) -> str:
    if pd.isna(value):
        return "<MISSING>"
    try:
        number = float(value)
    except (TypeError, ValueError):
        return str(value)
    return str(int(number)) if number.is_integer() else str(value)

normalized_labels = working[LABEL_COLUMNS].apply(lambda column: column.map(label_token))
normalized_labels["scope_target"] = scope_numeric.map(label_token)
for target_column in DIMENSIONS.values():
    normalized_labels.loc[scope_numeric.eq(0), target_column] = "N"
working["_label_signature"] = normalized_labels.agg("|".join, axis=1)

segment_keys = ["document_id", "segment_id"]
segment_audit = working.groupby(segment_keys, sort=False, dropna=False).agg(
    candidate_rows=("candidate_id", "size"),
    label_variants=("_label_signature", "nunique"),
    text_variants=("text", lambda values: values.nunique(dropna=False)),
    order_variants=("segment_order", lambda values: values.nunique(dropna=False)),
    invalid_candidate_rows=("_row_valid", lambda values: int((~values).sum())),
)
segment_audit["eligible"] = (
    segment_audit["label_variants"].eq(1)
    & segment_audit["text_variants"].eq(1)
    & segment_audit["order_variants"].eq(1)
    & segment_audit["invalid_candidate_rows"].eq(0)
)

order_segment_counts = working.groupby(["document_id", "segment_order"])["segment_id"].nunique()
colliding_orders = order_segment_counts[order_segment_counts.gt(1)]
order_collision_counts = colliding_orders.groupby(level="document_id").size()
document_audit = segment_audit.groupby(level="document_id").agg(
    segment_count=("candidate_rows", "size"),
    ineligible_segment_count=("eligible", lambda values: int((~values).sum())),
    candidate_row_count=("candidate_rows", "sum"),
)
document_audit["duplicate_segment_order_count"] = order_collision_counts.reindex(document_audit.index, fill_value=0)
document_audit["eligible"] = (
    document_audit["ineligible_segment_count"].eq(0)
    & document_audit["duplicate_segment_order_count"].eq(0)
)

def finding(rule: str, severity: str, count: int, unit: str) -> dict:
    return {"rule": rule, "severity": severity, "affected_count": int(count), "unit": unit}

rule_results = pd.DataFrame([
    finding("Unique candidate_id", "error", candidate_id_duplicates, "candidate rows"),
    finding("Text, component/scope, and label domains valid", "error", int((~working["_structural_valid"]).sum()), "candidate rows"),
    finding("Rows flagged requires_review", "review", int(working["_flag_requires_review"].sum()), "candidate rows"),
    finding("Candidate copies agree on segment text", "error", int(segment_audit["text_variants"].gt(1).sum()), "segments"),
    finding("Candidate copies agree on segment order", "error", int(segment_audit["order_variants"].gt(1).sum()), "segments"),
    finding("One complete label bundle per segment", "review", int(segment_audit["label_variants"].gt(1).sum()), "segments"),
    finding("Distinct segments have unique source order per document", "error", int(colliding_orders.size), "document/order pairs"),
    finding("All segments pass for whole-document exemplar use", "review", int((~document_audit["eligible"]).sum()), "documents"),
])
display(rule_results)
print(f"Automatically eligible whole documents: {int(document_audit['eligible'].sum()):,} of {len(document_audit):,}.")

,rule,severity,affected_count,unit
0,Unique candidate_id,error,0,candidate rows
1,"Text, component/scope, and label domains valid",error,0,candidate rows
2,Rows flagged requires_review,review,84,candidate rows
3,Candidate copies agree on segment text,error,0,segments
4,Candidate copies agree on segment order,error,0,segments
5,One complete label bundle per segment,review,32,segments
6,Distinct segments have unique source order per...,error,0,document/order pairs
7,All segments pass for whole-document exemplar use,review,30,documents


Automatically eligible whole documents: 151 of 181.


## 10. Generate Audit Summary Tables and Export Findings

Assemble the issue register and coverage summaries. Exports later in the notebook are optional, aggregate-only, and written outside the source tree; this cell does not expose example text or row identifiers.

In [15]:
quality_summary = pd.DataFrame([
    {"measure": "candidate rows", "count": len(working)},
    {"measure": "unique segments", "count": working["segment_id"].nunique()},
    {"measure": "training documents", "count": working["document_id"].nunique()},
    {"measure": "eligible documents", "count": int(document_audit["eligible"].sum())},
    {"measure": "documents needing expert review", "count": int((~document_audit["eligible"]).sum())},
])
display(quality_summary)

eligible_document_ids = set(document_audit.index[document_audit["eligible"]])
eligible_rows = working.loc[working["document_id"].isin(eligible_document_ids)].copy()
segment_consensus = eligible_rows.drop_duplicates(["document_id", "segment_id"], keep="first")
segment_consensus["scope_target"] = pd.to_numeric(segment_consensus["scope_target"], errors="raise").astype(int)

coverage_records = []
for label in (0, 1):
    coverage_records.append({"task": "scope", "label": str(label), "segment_count": int(segment_consensus["scope_target"].eq(label).sum())})
for dimension, column in DIMENSIONS.items():
    in_scope_segments = segment_consensus.loc[segment_consensus["scope_target"].eq(1)]
    for label in (0, 1, 2, 3):
        count = pd.to_numeric(in_scope_segments[column], errors="coerce").eq(label).sum()
        coverage_records.append({"task": dimension, "label": str(label), "segment_count": int(count)})
label_coverage = pd.DataFrame(coverage_records)
display(label_coverage.pivot(index="task", columns="label", values="segment_count").fillna(0).astype(int))

print("Coverage counts use one consistent label bundle per segment in eligible documents; support=0 means unavailable for exemplar coverage.")

,measure,count
0,candidate rows,7107
1,unique segments,7065
2,training documents,181
3,eligible documents,151
4,documents needing expert review,30


label,0,1,2,3
task,,,,
HA,4692,544,253,31
SW,3202,1880,421,17
UA,3146,1616,495,263
scope,292,5520,0,0


Coverage counts use one consistent label bundle per segment in eligible documents; support=0 means unavailable for exemplar coverage.


## 11. Select a Diverse Whole-Reflection Review Shortlist

A deterministic greedy selector favors documents that cover underrepresented labels, then uses shorter total text as a prompt-size tie-breaker. It selects complete reflections, not isolated segment snippets. The shortlist is only for expert review; automated consistency does not establish semantic correctness.

In [17]:
import secrets

# Build per-document coverage tags from the consistent, eligible segment table.
document_tags: dict[str, set[str]] = {}
for document_id, rows in segment_consensus.groupby("document_id", sort=True):
    tags = set()
    for _, row in rows.iterrows():
        tags.add(f"scope={int(row['scope_target'])}")
        if int(row["scope_target"]) == 1:
            for dimension, column in DIMENSIONS.items():
                value = int(row[column])
                tags.add(f"{dimension}={value}")
    document_tags[str(document_id)] = tags

document_sizes = segment_consensus.groupby("document_id").agg(
    segment_count=("segment_id", "nunique"),
    approximate_characters=("text", lambda values: int(values.astype("string").str.len().sum())),
)
document_frequency = {
    tag: sum(tag in tags for tags in document_tags.values())
    for tag in set().union(*document_tags.values())
} if document_tags else {}

selected_document_ids: list[str] = []
covered_tags: set[str] = set()
shortlist_rows = []
review_ref_to_document_id: dict[str, str] = {}
remaining = set(document_tags)
for selection_rank in range(1, min(MAX_SHORTLIST_DOCUMENTS, len(remaining)) + 1):
    scored = []
    for document_id in sorted(remaining):
        new_tags = document_tags[document_id] - covered_tags
        gain = sum(1 / np.sqrt(document_frequency[tag]) for tag in new_tags)
        char_count = max(int(document_sizes.loc[document_id, "approximate_characters"]), 1)
        scored.append((gain / np.sqrt(char_count), gain, -char_count, document_id, new_tags))
    best = max(scored, key=lambda item: (item[0], item[1], item[2], item[3]))
    if best[1] <= 0:
        break
    _, gain, neg_chars, document_id, new_tags = best
    selected_document_ids.append(document_id)
    covered_tags.update(document_tags[document_id])
    remaining.remove(document_id)
    review_ref = f"R-{secrets.token_hex(3)}"
    shortlist_rows.append({
        "review_ref": review_ref,
        "selection_rank": selection_rank,
        "segment_count": int(document_sizes.loc[document_id, "segment_count"]),
        "approximate_characters": -neg_chars,
        "new_label_strata_covered": int(len(new_tags)),
        "cumulative_label_strata_covered": int(len(covered_tags)),
    })
    review_ref_to_document_id[review_ref] = document_id

shortlist = pd.DataFrame(shortlist_rows)
display(shortlist)
print("Session-local refs are not stable IDs; rerunning creates new refs. No document IDs or text are shown.")
print(f"Covered strata: {len(covered_tags)} of {len(document_frequency)} present in the eligible pool.")

,review_ref,selection_rank,segment_count,approximate_characters,new_label_strata_covered,cumulative_label_strata_covered
0,R-6205c7,1,25,3506,12,12
1,R-5a4d40,2,22,3257,2,14


Session-local refs are not stable IDs; rerunning creates new refs. No document IDs or text are shown.
Covered strata: 14 of 14 present in the eligible pool.


## 12. Inventory Micro Labels and Sample Training Segments

Join the curated micro-label candidates to the audited training-only segment pool by `segment_id`. Keep only single-candidate, non-review rows with matching text and matching broad skill band. Skill codes are `SW` = Situationserfassung, `UA` = Analyse, and `HA` = Konsequenzen; Bands A/B/C map to original Stufen I-II/III/IV-V. For every observed `(skill, band, micro_label)` group, sample up to three distinct segments without replacement using a fixed seed. Smaller groups remain listed with their actual support, and the full 3-by-3 skill-band grid makes unavailable groups explicit. This is a segment-level review list, separate from the whole-reflection shortlist above.

In [27]:
MICRO_LABEL_SOURCE_PATH = (
    PROJECT_ROOT.parent
    / "prebi_ap4_v1"
    / "rl1_reflection_analytics/data/curated/prepared/segment_label_candidates.csv"
)
MICRO_SAMPLE_SEED = 20261004
MICRO_SAMPLE_PER_LABEL = 3
REVIEW_MICRO_TEXT = False
MICRO_REVIEW_REFS = []
SKILL_TO_TARGET_COLUMN = dict(DIMENSIONS)
LEVEL_TO_BAND = {
    "stufe i": 1, "stufe ii": 1,
    "stufe iii": 2,
    "stufe iv": 3, "stufe v": 3,
}
BAND_NAMES = {1: "Band A", 2: "Band B", 3: "Band C"}

if not MICRO_LABEL_SOURCE_PATH.is_file():
    raise FileNotFoundError("Curated micro-label candidate source was not found in the sibling prebi_ap4_v1 project.")
micro_required_columns = {
    "segment_id", "text_original", "category", "level", "micro_label",
    "candidate_count", "requires_review",
}
micro_source = pd.read_csv(
    MICRO_LABEL_SOURCE_PATH,
    usecols=lambda column: column in micro_required_columns,
    dtype={"segment_id": "string"},
)
missing_micro_columns = micro_required_columns - set(micro_source.columns)
if missing_micro_columns:
    raise ValueError(f"Micro-label source is missing required columns: {sorted(missing_micro_columns)}")
micro_source_rows = len(micro_source)
micro_source["candidate_count"] = pd.to_numeric(micro_source["candidate_count"], errors="coerce")
micro_source = micro_source.loc[
    micro_source["candidate_count"].eq(1)
    & ~as_bool(micro_source["requires_review"])
    & micro_source["micro_label"].notna()
    & micro_source["micro_label"].astype("string").str.strip().ne("")
].copy()
micro_source["micro_label"] = micro_source["micro_label"].astype("string").str.strip()
micro_source["skill"] = micro_source["category"].astype("string").str.strip().str.lower().map(
    {"situationserfassung": "SW", "analyse": "UA", "konsequenzen": "HA"}
)
micro_source["band"] = (
    micro_source["level"].astype("string").str.strip().str.lower()
    .map(LEVEL_TO_BAND)
)
micro_source = micro_source.loc[micro_source["skill"].notna() & micro_source["band"].notna()].copy()
micro_source["band"] = micro_source["band"].astype(int)
if micro_source["segment_id"].duplicated().any():
    raise ValueError("Single-candidate micro-label source has duplicate segment IDs.")

eligible_segment_keys = segment_audit.loc[segment_audit["eligible"]].reset_index()[
    ["document_id", "segment_id"]
]
micro_training_segments = (
    working.merge(eligible_segment_keys, on=["document_id", "segment_id"], how="inner", validate="many_to_one")
    .drop_duplicates(["document_id", "segment_id"])
)
if micro_training_segments["segment_id"].duplicated().any():
    raise ValueError("Training segment IDs are not globally unique for micro-label matching.")
micro_joined = micro_source.merge(
    micro_training_segments[["segment_id", "text", "scope_target", *DIMENSIONS.values()]],
    on="segment_id",
    how="inner",
    validate="one_to_one",
)
micro_joined["text_matches_training"] = (
    micro_joined["text_original"].astype("string").str.strip()
    == micro_joined["text"].astype("string").str.strip()
)
micro_joined["model_band"] = np.nan
for skill, target_column in SKILL_TO_TARGET_COLUMN.items():
    skill_mask = micro_joined["skill"].eq(skill)
    micro_joined.loc[skill_mask, "model_band"] = pd.to_numeric(
        micro_joined.loc[skill_mask, target_column], errors="coerce",
    )
micro_usable_mask = (
    micro_joined["text_matches_training"]
    & pd.to_numeric(micro_joined["scope_target"], errors="coerce").eq(1)
    & pd.to_numeric(micro_joined["model_band"], errors="coerce").eq(micro_joined["band"])
)
micro_usable = micro_joined.loc[micro_usable_mask].copy()
if micro_usable["segment_id"].duplicated().any():
    raise ValueError("Micro-label shortlist must contain at most one row per segment.")

micro_source_audit = pd.DataFrame([
    {"stage": "curated source", "count": int(micro_source_rows), "unit": "candidate rows"},
    {"stage": "single candidate, no review flag, known micro label", "count": int(len(micro_source)), "unit": "segments"},
    {"stage": "overlap with audited training segments", "count": int(len(micro_joined)), "unit": "segments"},
    {"stage": "exact text and broad-band match", "count": int(len(micro_usable)), "unit": "segments"},
])
display(micro_source_audit)

micro_inventory = (
    micro_usable.groupby(["skill", "band", "micro_label"], as_index=False)
    .agg(available_segments=("segment_id", "nunique"))
    .sort_values(["skill", "band", "micro_label"])
    .reset_index(drop=True)
)
rng = np.random.default_rng(MICRO_SAMPLE_SEED)
sampled_groups = []
for _, group in micro_usable.groupby(["skill", "band", "micro_label"], sort=True):
    sample_size = min(MICRO_SAMPLE_PER_LABEL, len(group))
    selected_indices = rng.choice(group.index.to_numpy(), size=sample_size, replace=False)
    sampled_groups.append(group.loc[selected_indices].copy())
micro_samples = pd.concat(sampled_groups, ignore_index=True) if sampled_groups else micro_usable.iloc[0:0].copy()
micro_samples = micro_samples.sort_values(["skill", "band", "micro_label", "segment_id"]).reset_index(drop=True)
if len(micro_samples) != micro_samples["segment_id"].nunique():
    raise ValueError("Random sampling selected a segment more than once.")
micro_samples["review_ref"] = [f"M-{secrets.token_hex(3)}" for _ in range(len(micro_samples))]
micro_review_lookup = {row["review_ref"]: row for _, row in micro_samples.iterrows()}
micro_inventory["sampled_segments"] = micro_inventory["available_segments"].clip(upper=MICRO_SAMPLE_PER_LABEL)
micro_inventory["band_name"] = micro_inventory["band"].map(BAND_NAMES)
print("Available micro labels by skill and broad band:")
with pd.option_context("display.max_colwidth", None):
    display(micro_inventory[["skill", "band_name", "micro_label", "available_segments", "sampled_segments"]])
print("Seeded segment shortlist; refs are session-local and no text or IDs are shown:")
micro_sample_list = micro_samples[["review_ref", "skill", "band", "micro_label"]].copy()
micro_sample_list["band_name"] = micro_sample_list["band"].map(BAND_NAMES)
micro_sample_list["sample_number"] = micro_sample_list.groupby(
    ["skill", "band", "micro_label"], sort=False,
).cumcount() + 1
with pd.option_context("display.max_colwidth", None):
    display(micro_sample_list[["review_ref", "skill", "band_name", "micro_label", "sample_number"]])

,stage,count,unit
0,curated source,11996,candidate rows
1,"single candidate, no review flag, known micro ...",4563,segments
2,overlap with audited training segments,3204,segments
3,exact text and broad-band match,3204,segments


Available micro labels by skill and broad band:


,skill,band_name,micro_label,available_segments,sampled_segments
0,HA,Band A,erkennen der notwendigkeit einer handlungsalternative,108,3
1,HA,Band A,keine nennung,1,1
2,HA,Band A,nennung von unplausiblen handlungsalternativen,21,3
3,HA,Band A,nennung zielführender handlungsalternativen (+),521,3
4,HA,Band A,schlussfolgerungen für zukünftige situationen (+) (+),33,3
5,UA,Band A,bewertung von qualitätsmerkmalen,235,3
6,UA,Band A,nennung von qualitätsmerkmalen,1127,3
7,UA,Band A,zielführende analyse ohne notw. anwendung von professionswissen,550,3
8,UA,Band B,begründung durch biographische bezüge,27,3
9,UA,Band B,begründung durch professionswissen,441,3


Seeded segment shortlist; refs are session-local and no text or IDs are shown:


,review_ref,skill,band_name,micro_label,sample_number
0,M-175dc6,HA,Band A,erkennen der notwendigkeit einer handlungsalternative,1
1,M-68af68,HA,Band A,erkennen der notwendigkeit einer handlungsalternative,2
2,M-6e220e,HA,Band A,erkennen der notwendigkeit einer handlungsalternative,3
3,M-ee7c87,HA,Band A,keine nennung,1
4,M-0d4647,HA,Band A,nennung von unplausiblen handlungsalternativen,1
5,M-78fd0f,HA,Band A,nennung von unplausiblen handlungsalternativen,2
6,M-1d8f46,HA,Band A,nennung von unplausiblen handlungsalternativen,3
7,M-8594ea,HA,Band A,nennung zielführender handlungsalternativen (+),1
8,M-b298ad,HA,Band A,nennung zielführender handlungsalternativen (+),2
9,M-42f15d,HA,Band A,nennung zielführender handlungsalternativen (+),3


In [28]:
micro_combo_grid = pd.MultiIndex.from_product(
    [["SW", "UA", "HA"], [1, 2, 3]], names=["skill", "band"],
).to_frame(index=False)
observed_combo_coverage = (
    micro_inventory.groupby(["skill", "band"], as_index=False)
    .agg(
        available_micro_labels=("micro_label", "nunique"),
        eligible_segments=("available_segments", "sum"),
        sampled_segments=("sampled_segments", "sum"),
    )
)
micro_combo_coverage = micro_combo_grid.merge(
    observed_combo_coverage, on=["skill", "band"], how="left", validate="one_to_one",
).fillna(0)
micro_combo_coverage["band_name"] = micro_combo_coverage["band"].map(BAND_NAMES)
for column in ["available_micro_labels", "eligible_segments", "sampled_segments"]:
    micro_combo_coverage[column] = micro_combo_coverage[column].astype(int)
display(micro_combo_coverage[[
    "skill", "band_name", "available_micro_labels", "eligible_segments", "sampled_segments",
]])

,skill,band_name,available_micro_labels,eligible_segments,sampled_segments
0,SW,Band A,0,0,0
1,SW,Band B,0,0,0
2,SW,Band C,0,0,0
3,UA,Band A,3,1912,9
4,UA,Band B,3,608,9
5,UA,Band C,0,0,0
6,HA,Band A,5,684,13
7,HA,Band B,0,0,0
8,HA,Band C,0,0,0


### Inspect Sampled Segment Text (Opt-In)

This cell renders the sampled segment text only when explicitly enabled and supplied current session-local refs. Outputs then contain controlled text; review locally and clear outputs before saving or sharing.

In [29]:
if REVIEW_MICRO_TEXT:
    if not MICRO_REVIEW_REFS:
        raise ValueError("Set MICRO_REVIEW_REFS to current refs from the micro-label shortlist.")
    unknown_micro_refs = set(MICRO_REVIEW_REFS) - set(micro_review_lookup)
    if unknown_micro_refs:
        raise ValueError("Micro review refs are invalid or stale; rerun the sampling cell and select current refs.")
    for review_ref in MICRO_REVIEW_REFS:
        sample = micro_review_lookup[review_ref]
        print(f"Protected segment review for {review_ref}; clear this output before saving/sharing.")
        display(pd.DataFrame([{
            "skill": sample["skill"],
            "band": int(sample["band"]),
            "micro_label": sample["micro_label"],
            "text": sample["text"],
        }]))
else:
    print("Micro-label text review is disabled. Set REVIEW_MICRO_TEXT=True and choose current MICRO_REVIEW_REFS in a protected local session.")

Micro-label text review is disabled. Set REVIEW_MICRO_TEXT=True and choose current MICRO_REVIEW_REFS in a protected local session.


## 13. Expert Review of Shortlisted Reflections

Before using any reflection or segment as a prompt example, an expert should verify that the labels are substantively defensible against the rubric and write a concise evidence note: (1) point to the relevant segment evidence, (2) name the rubric distinction that supports the scope/band/micro-label decision, and (3) note ambiguity or counterevidence. Do not generate explanations from labels alone, and do not request or store model chain-of-thought. Keep reviewer notes in approved protected storage, separate from this notebook.

In [18]:
if REVIEW_TEXT:
    if not REVIEW_REFS:
        raise ValueError("Set REVIEW_REFS to one or more session-local refs from the shortlist.")
    unknown_refs = set(REVIEW_REFS) - set(review_ref_to_document_id)
    if unknown_refs:
        raise ValueError("Review refs are invalid or stale; rerun the shortlist and select current refs.")
    for review_ref in REVIEW_REFS:
        document_id = review_ref_to_document_id[review_ref]
        document_rows = (
            segment_consensus.loc[segment_consensus["document_id"].eq(document_id)]
            .sort_values("segment_order")
        )
        safe_review = pd.DataFrame({
            "order": document_rows["segment_order"].astype(int),
            "text": document_rows["text"].astype(str),
            "scope": document_rows["scope_target"].astype(int).map({0: "out_of_scope", 1: "in_scope"}),
            **{
                dimension: document_rows[column].map(
                    lambda value: "N" if pd.isna(value) else str(int(value))
                )
                for dimension, column in DIMENSIONS.items()
            },
        })
        print(f"Protected full-text review for {review_ref}; clear this output before saving/sharing.")
        display(safe_review)
else:
    print("Full-text review is disabled. Set REVIEW_TEXT=True and REVIEW_REFS=[...] only in a protected local session.")

Full-text review is disabled. Set REVIEW_TEXT=True and REVIEW_REFS=[...] only in a protected local session.


## 14. Export Aggregate Findings (Optional)

Exports contain rule counts, broad-band coverage, and the available micro-label inventory only. They exclude text, candidate/document/segment identifiers, and review references. Keep exports disabled unless the destination is approved protected storage.

In [31]:
EXPORT_AGGREGATES = True
if EXPORT_AGGREGATES:
    if not PROTECTED_ROOT_VALUE:
        raise RuntimeError("Set PREBI_DATA_ROOT to approved protected storage before exporting.")
    protected_root = Path(PROTECTED_ROOT_VALUE).expanduser().resolve()
    if protected_root == PROJECT_ROOT or PROJECT_ROOT in protected_root.parents:
        raise ValueError("Protected storage must be outside the source tree.")
    export_root = protected_root / "gemini-exemplar-audit"
    export_root.mkdir(parents=True, exist_ok=True, mode=0o700)
    export_dir = export_root / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    export_dir.mkdir(mode=0o700, exist_ok=False)
    exports = {
        "audit_findings.csv": rule_results,
        "quality_summary.csv": quality_summary,
        "label_coverage.csv": label_coverage,
        "micro_label_inventory.csv": micro_inventory,
        "micro_label_skill_band_coverage.csv": micro_combo_coverage,
    }
    for filename, table in exports.items():
        target = export_dir / filename
        table.to_csv(target, index=False)
        os.chmod(target, 0o600)
    print("Aggregate audit tables exported to approved protected storage; path withheld.")
else:
    print("Aggregate export disabled; no files were written.")

Aggregate audit tables exported to approved protected storage; path withheld.


## Interpretation and Next Steps

- An automatically eligible document has internally consistent labels and clean structural checks; this does not prove that its labels are correct.
- The split export contains no annotation rationales. Exemplar explanations must be authored and reviewed by a qualified human using the rubric and text evidence.
- Freeze the exemplar set and prompt version before evaluating on the held-out test set. Compare no-example, example-only, and example-plus-evidence-note prompts on dev first, holding model settings and scoring constant.
- Preserve candidate annotations and disagreement records. Do not use test reflections to choose examples, explanations, or prompt wording.

## 15. Export the Selected Micro-Label Sample List

This separate export contains controlled segment text and pseudonymous segment IDs for expert review. It is written only beneath approved `PREBI_DATA_ROOT`, in a unique directory with restrictive permissions. Do not move it into the source tree or share it outside the approved review workflow.

In [32]:
EXPORT_MICRO_SAMPLE_LIST = True
if EXPORT_MICRO_SAMPLE_LIST:
    if micro_samples.empty:
        raise RuntimeError("There are no eligible micro-label samples to export.")
    if not PROTECTED_ROOT_VALUE:
        raise RuntimeError("Set PREBI_DATA_ROOT to approved protected storage before exporting samples.")
    protected_root = Path(PROTECTED_ROOT_VALUE).expanduser().resolve()
    if protected_root == PROJECT_ROOT or PROJECT_ROOT in protected_root.parents:
        raise ValueError("Protected storage must be outside the source tree.")
    sample_export_root = protected_root / "gemini-exemplar-audit" / "micro-label-samples"
    sample_export_root.mkdir(parents=True, exist_ok=True, mode=0o700)
    os.chmod(sample_export_root, 0o700)
    sample_export_dir = sample_export_root / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    sample_export_dir.mkdir(mode=0o700, exist_ok=False)

    sample_export = micro_samples[[
        "segment_id", "skill", "band", "micro_label", "text",
    ]].copy()
    sample_export["skill_name"] = sample_export["skill"].map({
        "SW": "Situationserfassung", "UA": "Analyse", "HA": "Konsequenzen",
    })
    sample_export["band_name"] = sample_export["band"].map(BAND_NAMES)
    sample_export["sample_number"] = sample_export.groupby(
        ["skill", "band", "micro_label"], sort=False,
    ).cumcount() + 1
    sample_export = sample_export[[
        "segment_id", "skill", "skill_name", "band_name", "micro_label", "sample_number", "text",
    ]]
    sample_path = sample_export_dir / "micro-label-segment-samples.csv"
    sample_export.to_csv(sample_path, index=False)
    os.chmod(sample_path, 0o600)
    print(f"Exported {len(sample_export)} sampled segments across {micro_inventory.shape[0]} micro-label groups to protected storage; path withheld.")
else:
    print("Micro-label sample export disabled; no files were written.")

Exported 31 sampled segments across 11 micro-label groups to protected storage; path withheld.
